# BioNNE-L: Cross-Encoder Reranking over a Fine-Tuned Dense Retriever

This notebook trains a cross-encoder reranker on top of the dense retriever fine-tuned in `bionnel-dense-finetuning.ipynb`. It supports BCE, ListNet, and LambdaLoss objectives, selects the best checkpoint by dev-set reranking quality, and writes final prediction artifacts.


In [ ]:
import copy
import gc
import json
import logging
import os
import random
import shutil
from pathlib import Path

import mlflow
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from lib.data.text_preprocessing import preprocess_text
from lib.data.vocab_enrichment import (
    enrich_vocab_with_oov_train_dev_terms,
    filter_vocab_for_dataset_language,
    prepare_experiment_vocab,
)
from lib.retrieval.reranking.candidate_context import build_candidate_context_cache
from lib.retrieval.reranking.candidate_context_cache import (
    build_candidate_text_map,
    load_candidate_context_cache,
    save_candidate_context_cache,
)
from lib.retrieval.reranking.context import build_contextualized_mentions, summarize_context_coverage
from lib.retrieval.reranking.dictionary_pretrain.artifacts import build_dictionary_pretrain_cache_metadata
from lib.retrieval.reranking.dictionary_pretrain.fingerprints import fingerprint_dictionary_pretrain_dataframe
from lib.retrieval.reranking.candidate_cache import (
    build_retriever_candidate_cache,
    load_retriever_candidate_cache,
    save_retriever_candidate_cache,
)
from lib.retrieval.reranking.inference import rerank_from_candidate_cache
from lib.retrieval.reranking.io import load_cross_encoder_model_with_config, save_json
from lib.retrieval.reranking.training import train_cross_encoder_reranker
from lib.retrieval.tuning import evaluate_dev_predictions
from lib.utils.logging_utils import configure_logging


In [ ]:
configure_logging(level=logging.INFO, force=True)

ARTIFACTS_DIR = './artifacts_cross_encoder_reranking'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

MLFLOW_EXPERIMENT_PREFIX = 'bionnel'
MLFLOW_RUN_NAME_TEMPLATE = '{dataset_name}-cross-encoder-reranking-lambda-pretrained-candidate-context'

DEFAULT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEFAULT_DEVICE

In [ ]:
LOCAL_MLRUNS_DIR = Path('mlruns').resolve()
mlflow.set_tracking_uri(LOCAL_MLRUNS_DIR.as_uri())
print('MLflow tracking URI:', mlflow.get_tracking_uri())


In [ ]:
## Common Hyperparameters
COMMON_TRAINING_CONFIG = {
    'SEED': 42,  # Random seed for splitting, sampling, and training.
    'EPOCHS': 5,  # Number of supervised reranker fine-tuning epochs.
    'TRAIN_BATCH_SIZE': 32,  # Per-device training batch size for cross-encoder pairs.
    'EVAL_BATCH_SIZE': 32,  # Per-device batch size for dev-loss evaluation.
    'LEARNING_RATE': 1e-5,  # Optimizer learning rate.
    'WEIGHT_DECAY': 0.01,  # Optimizer weight decay.
    'WARMUP_RATIO': 0.1,  # Warmup fraction for the learning-rate schedule.
    'LR_SCHEDULER_TYPE': 'linear',  # Learning-rate scheduler: 'linear' or 'cosine'.
    'USE_FP16': False,  # Enable fp16 mixed precision.
    'USE_BF16': False,  # Enable bf16 mixed precision.
    'ALLOW_TF32': False,  # Allow TF32 CUDA kernels.
    'MAX_SEQ_LENGTH': 384,  # Maximum token length for mention-candidate pairs.
    'GRAD_ACCUMULATION_STEPS': 1,  # Number of batches accumulated before an optimizer step.
    'TRAIN_LOGGING_STEPS': 25,  # Training log interval in optimizer steps.
    'DEV_LOSS_EVAL_STEPS': 0,  # Dev-loss interval; 0 evaluates by epoch.
    'EVAL_EVERY_EPOCH': True,  # Run dev reranking evaluation after each epoch.
    'EARLY_STOPPING_PATIENCE': 3,  # Stop after this many non-improving dev evaluations.
    'SELECTION_METRIC': 'Acc@1',  # Checkpoint selection metric: 'Acc@1' or 'MRR'.
    'LOSS_NAME': 'lambdaloss',  # Training objective: 'bce', 'listnet', or 'lambdaloss'.
    'LAMBDALOSS_K': None,  # Optional top-k truncation for LambdaLoss; None uses the full list.
    'LAMBDALOSS_WEIGHTING_SCHEME': 'ndcg2pp',  # LambdaLoss weighting: 'ndcg2pp', 'ndcg2', 'ndcg1', 'lambdarank', or 'none'.
    'LAMBDALOSS_SIGMA': 1.0,  # Pairwise score-difference scale for LambdaLoss.
    'LAMBDALOSS_REDUCTION_LOG': 'binary',  # LambdaLoss logarithm base: 'binary' or 'natural'.
    'LAMBDALOSS_MINI_BATCH_SIZE': None,  # Optional internal chunk size for long candidate lists.
    'RERANK_BATCH_SIZE': 64,  # CrossEncoder.predict batch size during reranking evaluation/inference.
}

COMMON_MODEL_CONFIG = {
    'TRUST_REMOTE_CODE': False,  # Allow custom Hugging Face model code when loading checkpoints.
    'LOCAL_FILES_ONLY': False,  # Load models only from the local Hugging Face cache.
    'TORCH_DTYPE': None,  # Model dtype: None, 'auto', 'float16', 'bfloat16', or 'float32'.
}

COMMON_RETRIEVAL_CONFIG = {
    'ENRICH_VOCABULARY': False,  # Add all unique train/dev mention-CUI pairs beyond default OOV-only enrichment.
    'TEST_ENRICH_VOCABULARY': True,  # Apply the same train/dev-only enrichment during final test inference.
    'DEDUPLICATE_BY_CUI': True,  # Keep at most one retriever candidate per CUI.
    'TRAIN_CANDIDATE_POOL_SIZE': 20,  # Retriever candidates kept for each training mention.
    'DEV_CANDIDATE_POOL_SIZE': 20,  # Retriever candidates scored by the reranker on dev.
    'DEV_RETURN_TOPK': 20,  # Reranked candidates kept in dev predictions and metrics.
    'TEST_CANDIDATE_POOL_SIZE': 20,  # Retriever candidates scored by the reranker on test.
    'TEST_RETURN_TOPK': 5,  # Final number of test predictions written per mention.
    'QUERY_BATCH_SIZE': 262_144,  # Retriever query batch size for candidate-cache construction.
    'DENSE_VOCAB_BATCH_SIZE': 16_384,  # Vocabulary chunk size for dense retriever scoring.
    'ST_ENCODE_BATCH_SIZE': 1024,  # SentenceTransformer encoding batch size for retriever inputs.
    'LOAD_FROM_DISK_IF_AVAILABLE': True,  # Reuse retriever candidate caches when metadata matches.
    'FORCE_REBUILD_TRAIN_CACHE': False,  # Rebuild the train retriever candidate cache.
    'FORCE_REBUILD_DEV_CACHE': False,  # Rebuild the dev retriever candidate cache.
    'FORCE_REBUILD_TEST_CACHE': False,  # Rebuild the test retriever candidate cache.
    'TRAIN_CACHE_STEM': 'train_retriever_cache',  # File stem for train retriever-cache artifacts.
    'DEV_CACHE_STEM': 'dev_retriever_cache',  # File stem for dev retriever-cache artifacts.
    'TEST_CACHE_STEM': 'test_retriever_cache',  # File stem for test retriever-cache artifacts.
}

COMMON_MENTION_CONTEXT_CONFIG = {
    'ENABLED': False,  # Add document/nesting context to mention strings.
    'MODE': 'hybrid',  # Mention context mode: 'nested_entities', 'text_window', or 'hybrid'.
    'FORMAT': 'explicit_markers',  # Mention context format: 'sep_token' or 'explicit_markers'.
    'DROP_NON_NESTED': False,  # Drop mentions without nested-entity context in nested_entities mode.
    'WINDOW_WORDS': 8,  # Symmetric text-window size in words.
    'LEFT_WINDOW_WORDS': None,  # Optional left-side text-window size override.
    'RIGHT_WINDOW_WORDS': None,  # Optional right-side text-window size override.
    'HYBRID_WINDOW_WORDS': None,  # Optional symmetric fallback window for hybrid mode.
    'HYBRID_LEFT_WINDOW_WORDS': 4,  # Left fallback window size for hybrid mode.
    'HYBRID_RIGHT_WINDOW_WORDS': 4,  # Right fallback window size for hybrid mode.
    'TEXTS_ROOT': 'data/texts',  # Root directory with source documents for mention context.
    'TEXT_DIR_TEMPLATE': None,  # Optional per-language text directory template.
    'CONTEXTUALIZED_MENTION_COLUMN': 'contextualized_mention_text',  # Column storing formatted mention context.
    'SEP_TOKEN': None,  # Optional separator token for sep-token context formatting.
}

COMMON_CANDIDATE_CONTEXT_CONFIG = {
    'ENABLED': True,  # Use heuristic candidate-context profiles instead of raw candidate names.
    'ALIAS_LENGTH_THRESHOLD': None,  # If None, auto-estimate alias length cap from vocabulary statistics.
    'MAX_ALIASES': 6,  # Maximum number of aliases added to each candidate profile.
    'GROUP_LIMITS': {
        'abbreviations': 1,  # Maximum abbreviation aliases per candidate.
        'short_names': 2,  # Maximum short-name aliases per candidate.
        'multi_word': 2,  # Maximum multi-word aliases per candidate.
        'long_variants': 1,  # Maximum long-form aliases per candidate.
    },
    'PREFERRED_LANGUAGES': None,  # Preferred alias languages; None uses dataset defaults.
    'ALLOWED_LANGUAGES': [],  # Allowed alias languages; empty list keeps dataset defaults.
    'NUM_WORKERS': 4,  # Parallel workers for candidate-context construction.
    'LOAD_FROM_DISK_IF_AVAILABLE': True,  # Reuse candidate-context cache when metadata matches.
    'FORCE_REBUILD': False,  # Rebuild candidate-context cache even when a cache exists.
    'CACHE_STEM': 'candidate_context_short',  # File stem for candidate-context cache artifacts.
}


## Data Loading


In [ ]:
ru_data_train = pd.read_parquet('data/parquet/ru/bionnel_ru_train.parquet')
ru_data_dev = pd.read_parquet('data/parquet/ru/bionnel_ru_dev.parquet')
ru_data_test = pd.read_csv('data/tsv/ru/bionnel_ru_test.tsv', sep='	')

en_data_train = pd.read_parquet('data/parquet/en/bionnel_en_train.parquet')
en_data_dev = pd.read_parquet('data/parquet/en/bionnel_en_dev.parquet')
en_data_test = pd.read_csv('data/tsv/en/bionnel_en_test.tsv', sep='	')

bilingual_data_train = pd.read_parquet('data/parquet/bilingual/bionnel_bilingual_train.parquet')
bilingual_data_dev = pd.read_parquet('data/parquet/bilingual/bionnel_bilingual_dev.parquet')
bilingual_data_test = pd.read_csv('data/tsv/bilingual/bionnel_bilingual_test.tsv', sep='	')

raw_vocab = pd.read_parquet('data/vocabular/bionnel_vocab_bilingual.parquet')
vocab = raw_vocab.copy()

for dataset_df in [
    ru_data_train,
    ru_data_dev,
    ru_data_test,
    en_data_train,
    en_data_dev,
    en_data_test,
    bilingual_data_train,
    bilingual_data_dev,
    bilingual_data_test,
]:
    dataset_df['raw_text'] = dataset_df['text'].astype(str)
    dataset_df['text'] = dataset_df['text'].map(preprocess_text)

vocab['concept_name'] = vocab['concept_name'].map(preprocess_text)

RU_ENTITIES_FOR_VOCAB_ENRICHMENT = pd.concat(
    [
        ru_data_train,
        ru_data_dev,
        en_data_train,
        en_data_dev,
        bilingual_data_train,
        bilingual_data_dev,
    ],
    ignore_index=True,
)
RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT = pd.concat(
    [
        ru_data_train,
        ru_data_dev,
    ],
    ignore_index=True,
)
EN_ENTITIES_FOR_VOCAB_ENRICHMENT = pd.concat(
    [
        en_data_train,
        en_data_dev,
    ],
    ignore_index=True,
)
EN_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT = pd.concat(
    [
        en_data_train,
        en_data_dev,
    ],
    ignore_index=True,
)

BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT = RU_ENTITIES_FOR_VOCAB_ENRICHMENT.copy()
BILINGUAL_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT = pd.concat(
    [
        bilingual_data_train,
        bilingual_data_dev,
    ],
    ignore_index=True,
)

print('Base vocabulary shape:', vocab.shape)

print('RU train/dev/test:', ru_data_train.shape, ru_data_dev.shape, ru_data_test.shape)
print('EN train/dev/test:', en_data_train.shape, en_data_dev.shape, en_data_test.shape)
print('Bilingual train/dev/test:', bilingual_data_train.shape, bilingual_data_dev.shape, bilingual_data_test.shape)
print('RU vocab enrichment pool:', RU_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print('RU raw candidate-context enrichment pool:', RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT.shape)
print('EN-only vocab enrichment pool:', EN_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print('EN raw candidate-context enrichment pool:', EN_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT.shape)
print('Bilingual vocab enrichment pool:', BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print('Bilingual raw candidate-context enrichment pool:', BILINGUAL_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT.shape)
print('Semantic types:', sorted(vocab['semantic_type'].dropna().unique().tolist()))


## Training Utils


In [ ]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



def get_artifact_dir(dataset_name):
    artifact_dir = Path(ARTIFACTS_DIR) / dataset_name.lower()
    artifact_dir.mkdir(parents=True, exist_ok=True)
    return artifact_dir



def resolve_default_retriever_model_path(dataset_name):
    return get_artifact_dir(dataset_name).parent.parent / 'artifacts_dense_finetuning' / dataset_name.lower() / 'training' / 'best_model'



def build_base_experiment_config(reranker_model_name_or_path, dataset_name, retriever_model_path=None):
    retriever_model_path = retriever_model_path or resolve_default_retriever_model_path(dataset_name)
    return {
        'RERANKER_MODEL_NAME_OR_PATH': reranker_model_name_or_path,
        'DEVICE': DEFAULT_DEVICE,
        'TRAINING': copy.deepcopy(COMMON_TRAINING_CONFIG),
        'MODEL': copy.deepcopy(COMMON_MODEL_CONFIG),
        'RETRIEVAL': copy.deepcopy(COMMON_RETRIEVAL_CONFIG),
        'MENTION_CONTEXT': copy.deepcopy(COMMON_MENTION_CONTEXT_CONFIG),
        'CANDIDATE_CONTEXT': copy.deepcopy(COMMON_CANDIDATE_CONTEXT_CONFIG),
        'RETRIEVER_MODEL_NAME_OR_PATH': str(retriever_model_path),
    }



def build_runtime_config(cfg):
    runtime_cfg = {
        'RERANKER_MODEL_NAME_OR_PATH': cfg['RERANKER_MODEL_NAME_OR_PATH'],
        'DEVICE': cfg['DEVICE'],
        'RETRIEVER_MODEL_NAME_OR_PATH': cfg['RETRIEVER_MODEL_NAME_OR_PATH'],
    }
    runtime_cfg.update(copy.deepcopy(cfg['TRAINING']))
    runtime_cfg.update(copy.deepcopy(cfg['RETRIEVAL']))
    runtime_cfg['TRAINING'] = copy.deepcopy(cfg['TRAINING'])
    runtime_cfg['MODEL'] = copy.deepcopy(cfg['MODEL'])
    runtime_cfg['RETRIEVAL'] = copy.deepcopy(cfg['RETRIEVAL'])
    runtime_cfg['MENTION_CONTEXT'] = copy.deepcopy(cfg['MENTION_CONTEXT'])
    runtime_cfg.update(runtime_cfg['MODEL'])
    runtime_cfg['CANDIDATE_CONTEXT'] = copy.deepcopy(cfg['CANDIDATE_CONTEXT'])
    return runtime_cfg



def build_test_retrieval_config(cfg):
    test_cfg = copy.deepcopy(cfg)
    test_cfg['RETRIEVAL']['ENRICH_VOCABULARY'] = bool(
        cfg['RETRIEVAL'].get('TEST_ENRICH_VOCABULARY', cfg['RETRIEVAL'].get('ENRICH_VOCABULARY', False))
    )
    return test_cfg



def build_test_retrieval_vocab(base_vocab_df, enrichment_entities_df, cfg, *, dataset_name=None, lang_value=None):
    test_cfg = build_test_retrieval_config(cfg)
    test_vocab_df = enrich_vocab_with_oov_train_dev_terms(
        base_vocab_df.copy(),
        enrichment_entities_df,
        lang_value=lang_value,
    )
    test_vocab_df = prepare_experiment_vocab(
        test_vocab_df,
        enrichment_entities_df,
        test_cfg['RETRIEVAL'],
        lang_value=lang_value,
    )
    if dataset_name is not None:
        test_vocab_df = filter_vocab_for_dataset_language(test_vocab_df, dataset_name)
    return test_vocab_df



def build_effective_experiment_config(base_cfg, training_result, prepared_summaries, candidate_artifact_paths, candidate_metadata, retriever_cache_paths, retriever_cache_metadata):
    effective_cfg = copy.deepcopy(base_cfg)
    effective_cfg['TRAINING'].update({
        'LOSS_NAME': str(base_cfg['TRAINING'].get('LOSS_NAME', 'bce')),
        'BEST_EPOCH': int(training_result['best_epoch']),
        'BEST_MODEL_DIR': str(training_result['best_model_dir']),
        'BEST_CHECKPOINT_DIR': str(training_result['best_checkpoint_dir']),
        'NUM_TRAIN_LISTS': int(len(training_result['train_examples_df'])),
        'NUM_TRAIN_PAIRS': int(len(training_result.get('train_pair_examples_df', []))),
    })
    effective_cfg['MENTION_CONTEXT']['PREPARED_SPLIT_SUMMARIES'] = prepared_summaries
    effective_cfg['CANDIDATE_CONTEXT']['ARTIFACTS'] = candidate_artifact_paths
    effective_cfg['CANDIDATE_CONTEXT']['METADATA'] = candidate_metadata
    effective_cfg['RETRIEVAL']['CACHE_ARTIFACTS'] = retriever_cache_paths
    effective_cfg['RETRIEVAL']['CACHE_METADATA'] = retriever_cache_metadata
    return effective_cfg



def resolve_candidate_context_languages(dataset_name, cfg):
    configured_languages = cfg['CANDIDATE_CONTEXT'].get('PREFERRED_LANGUAGES')
    if configured_languages:
        return [str(language).upper() for language in configured_languages]

    dataset_name = str(dataset_name).lower()
    if dataset_name == 'ru':
        return ['RUS', 'ENG']
    if dataset_name == 'en':
        return ['ENG', 'RUS']
    return ['RUS', 'ENG']



def resolve_candidate_context_allowed_languages(dataset_name, cfg):
    configured_languages = cfg['CANDIDATE_CONTEXT'].get('ALLOWED_LANGUAGES')
    if configured_languages:
        return [str(language).upper() for language in configured_languages]

    dataset_name = str(dataset_name).lower()
    if dataset_name == 'en':
        return ['ENG']
    return []



def load_retriever_model(model_name_or_path, device=None):
    model_path = Path(model_name_or_path)
    resolved_model = str(model_path) if model_path.exists() else str(model_name_or_path)
    resolved_device = DEFAULT_DEVICE if device is None else device
    return SentenceTransformer(resolved_model, device=resolved_device)



def prepare_mention_split(df, cfg, split_name):
    context_cfg = cfg['MENTION_CONTEXT']
    if not context_cfg.get('ENABLED', False):
        result_df = df.copy()
        result_df[context_cfg.get('CONTEXTUALIZED_MENTION_COLUMN', 'contextualized_mention_text')] = result_df['text']
        return result_df, {
            'enabled': False,
            'split_name': split_name,
            'context_column': context_cfg.get('CONTEXTUALIZED_MENTION_COLUMN', 'contextualized_mention_text'),
            'coverage': 0.0,
        }

    prepared_df = build_contextualized_mentions(
        entities_df=df,
        context_mode=context_cfg['MODE'],
        context_format=context_cfg['FORMAT'],
        sep_token=context_cfg['SEP_TOKEN'],
        drop_non_nested=context_cfg.get('DROP_NON_NESTED', False),
        split_name=split_name,
        texts_root=context_cfg.get('TEXTS_ROOT', 'data/texts'),
        text_dir_template=context_cfg.get('TEXT_DIR_TEMPLATE'),
        window_words=context_cfg.get('WINDOW_WORDS'),
        left_window_words=context_cfg.get('LEFT_WINDOW_WORDS'),
        right_window_words=context_cfg.get('RIGHT_WINDOW_WORDS'),
        hybrid_window_words=context_cfg.get('HYBRID_WINDOW_WORDS'),
        hybrid_left_window_words=context_cfg.get('HYBRID_LEFT_WINDOW_WORDS'),
        hybrid_right_window_words=context_cfg.get('HYBRID_RIGHT_WINDOW_WORDS'),
        text_preprocessor=preprocess_text,
    )
    return prepared_df, summarize_context_coverage(prepared_df)



def prepare_candidate_context_vocab(raw_base_vocab_df, enrichment_entities_df, cfg, *, lang_value=None):
    return prepare_experiment_vocab(
        raw_base_vocab_df,
        enrichment_entities_df,
        cfg['RETRIEVAL'],
        text_column='raw_text',
        lang_value=lang_value,
    )



def _load_cache_metadata(metadata_path):
    metadata_path = Path(metadata_path)
    if not metadata_path.exists():
        return None
    return json.loads(metadata_path.read_text(encoding='utf-8'))



def _can_reuse_cache(metadata_path, expected_metadata):
    existing_metadata = _load_cache_metadata(metadata_path)
    return existing_metadata == expected_metadata



def _can_reuse_retriever_cache(metadata_path, expected_metadata, requested_topk):
    existing_metadata = _load_cache_metadata(metadata_path)
    if existing_metadata is None:
        return False, None

    existing_cfg = dict(existing_metadata.get('config', {}))
    expected_cfg = dict(expected_metadata.get('config', {}))
    existing_topk = int(existing_cfg.pop('topk', 0) or 0)
    expected_topk = int(expected_cfg.pop('topk', 0) or 0)

    if existing_metadata.get('vocab_fingerprint') != expected_metadata.get('vocab_fingerprint'):
        return False, existing_metadata
    if existing_cfg != expected_cfg:
        return False, existing_metadata
    if existing_topk < int(requested_topk):
        return False, existing_metadata
    if existing_topk < expected_topk:
        return False, existing_metadata
    return True, existing_metadata



def _infer_retriever_candidate_cache_topk(cache):
    max_topk = 0
    for payload in cache.values():
        candidate_scores = payload.get('candidate_scores')
        if candidate_scores is None:
            continue
        if hasattr(candidate_scores, 'shape') and len(candidate_scores.shape) == 2:
            max_topk = max(max_topk, int(candidate_scores.shape[1]))
            continue
        if len(candidate_scores) > 0:
            max_topk = max(max_topk, int(len(candidate_scores[0])))
    return int(max_topk)



def _trim_retriever_candidate_cache(cache, *, topk):
    trimmed_cache = {}
    topk = int(topk)
    for entity_type, payload in cache.items():
        trimmed_payload = dict(payload)
        candidate_scores = payload.get('candidate_scores')
        candidate_indices = payload.get('candidate_indices')
        if candidate_scores is not None:
            trimmed_payload['candidate_scores'] = candidate_scores[:, :topk]
        if candidate_indices is not None:
            trimmed_payload['candidate_indices'] = candidate_indices[:, :topk]
        trimmed_cache[entity_type] = trimmed_payload
    return trimmed_cache



def _fingerprint_retriever_queries(entities_df, *, mention_column):
    fingerprint_columns = [
        mention_column,
        'document_id',
        'spans',
        'entity_type',
    ]
    if 'UMLS_CUI' in entities_df.columns:
        fingerprint_columns.append('UMLS_CUI')
    return fingerprint_dictionary_pretrain_dataframe(entities_df, columns=fingerprint_columns)



def _build_retriever_cache_expected_metadata(vocab_df, entities_df, cfg, *, mention_column, topk, cache_stem):
    return build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'retriever_model_name_or_path': cfg['RETRIEVER_MODEL_NAME_OR_PATH'],
            'mention_column': str(mention_column),
            'topk': int(topk),
            'deduplicate_by_cui': bool(cfg['DEDUPLICATE_BY_CUI']),
            'queries_fingerprint': _fingerprint_retriever_queries(entities_df, mention_column=mention_column),
            'cache_stem': str(cache_stem),
        },
    )



def prepare_candidate_context_cache_for_experiment(vocab_df, dataset_name, cfg):
    if not cfg['CANDIDATE_CONTEXT'].get('ENABLED', False):
        return None, {}, {}, {}

    artifact_dir = get_artifact_dir(dataset_name)
    cache_stem = cfg['CANDIDATE_CONTEXT'].get('CACHE_STEM', 'candidate_context')
    cache_path = artifact_dir / f'{cache_stem}.parquet'
    metadata_path = artifact_dir / f'{cache_stem}_metadata.json'

    expected_metadata = build_dictionary_pretrain_cache_metadata(
        vocab_df=vocab_df,
        cfg={
            'alias_length_threshold': cfg['CANDIDATE_CONTEXT'].get('ALIAS_LENGTH_THRESHOLD'),
            'max_aliases': cfg['CANDIDATE_CONTEXT'].get('MAX_ALIASES', 8),
            'group_limits': cfg['CANDIDATE_CONTEXT'].get('GROUP_LIMITS'),
            'preferred_languages': resolve_candidate_context_languages(dataset_name, cfg),
            'allowed_languages': resolve_candidate_context_allowed_languages(dataset_name, cfg),
            'cache_stem': cache_stem,
        },
    )

    use_disk_cache = (
        cfg['CANDIDATE_CONTEXT'].get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not cfg['CANDIDATE_CONTEXT'].get('FORCE_REBUILD', False)
        and cache_path.exists()
        and _can_reuse_cache(metadata_path, expected_metadata)
    )

    if use_disk_cache:
        candidate_context_df, _ = load_candidate_context_cache(cache_path, metadata_path)
        artifact_paths = {
            'cache_path': str(cache_path),
            'preview_path': str(artifact_dir / f'{cache_stem}_preview.parquet'),
            'metadata_path': str(metadata_path),
        }
    else:
        candidate_context_df, _ = build_candidate_context_cache(
            vocab_df=vocab_df,
            alias_length_threshold=cfg['CANDIDATE_CONTEXT'].get('ALIAS_LENGTH_THRESHOLD'),
            max_aliases=cfg['CANDIDATE_CONTEXT'].get('MAX_ALIASES', 8),
            group_limits=cfg['CANDIDATE_CONTEXT'].get('GROUP_LIMITS'),
            preferred_languages=resolve_candidate_context_languages(dataset_name, cfg),
            allowed_languages=resolve_candidate_context_allowed_languages(dataset_name, cfg),
            num_workers=cfg['CANDIDATE_CONTEXT'].get('NUM_WORKERS', 1),
        )
        artifact_paths = save_candidate_context_cache(
            candidate_context_df,
            expected_metadata,
            artifact_dir,
            stem=cache_stem,
        )

    candidate_metadata = dict(expected_metadata)
    if candidate_context_df is not None and not candidate_context_df.empty and 'alias_length_threshold' in candidate_context_df.columns:
        threshold_value = candidate_context_df['alias_length_threshold'].iloc[0]
        candidate_metadata['threshold'] = None if pd.isna(threshold_value) else int(threshold_value)
    candidate_metadata['num_candidate_rows'] = 0 if candidate_context_df is None else int(len(candidate_context_df))
    candidate_metadata['loaded_from_disk'] = bool(use_disk_cache)
    candidate_text_map = build_candidate_text_map(candidate_context_df)
    return candidate_context_df, candidate_metadata, artifact_paths, candidate_text_map



def prepare_retriever_candidate_cache(dataset_name, split_name, entities_df, vocab_df, cfg, retriever_model, *, topk, mention_column='text'):
    artifact_dir = get_artifact_dir(dataset_name)
    retrieval_cfg = cfg['RETRIEVAL']
    split_name = str(split_name).lower()
    stem = retrieval_cfg.get(f'{split_name.upper()}_CACHE_STEM', f'{split_name}_retriever_cache')
    cache_path = artifact_dir / f'{stem}.pkl'
    preview_path = artifact_dir / f'{stem}.tsv'
    metadata_path = artifact_dir / f'{stem}_metadata.json'
    expected_metadata = _build_retriever_cache_expected_metadata(
        vocab_df=vocab_df,
        entities_df=entities_df,
        cfg=cfg,
        mention_column=mention_column,
        topk=topk,
        cache_stem=stem,
    )

    use_disk_cache = False
    existing_metadata = None
    if (
        retrieval_cfg.get('LOAD_FROM_DISK_IF_AVAILABLE', True)
        and not retrieval_cfg.get(f'FORCE_REBUILD_{split_name.upper()}_CACHE', False)
        and cache_path.exists()
    ):
        use_disk_cache, existing_metadata = _can_reuse_retriever_cache(
            metadata_path,
            expected_metadata,
            topk,
        )

    if use_disk_cache:
        retriever_cache = load_retriever_candidate_cache(cache_path)
        cached_topk = _infer_retriever_candidate_cache_topk(retriever_cache)
        if cached_topk > int(topk):
            retriever_cache = _trim_retriever_candidate_cache(retriever_cache, topk=topk)
    else:
        retriever_cache = build_retriever_candidate_cache(
            entities_df=entities_df,
            vocab_df=vocab_df,
            retriever_model=retriever_model,
            mention_column=mention_column,
            topk=topk,
            query_batch_size=cfg['QUERY_BATCH_SIZE'],
            dense_vocab_batch_size=cfg['DENSE_VOCAB_BATCH_SIZE'],
            st_encode_batch_size=cfg['ST_ENCODE_BATCH_SIZE'],
            deduplicate_by_cui=cfg['DEDUPLICATE_BY_CUI'],
        )
        save_retriever_candidate_cache(retriever_cache, artifact_dir, stem=stem)
        save_json(expected_metadata, metadata_path)

    metadata = dict(expected_metadata)
    metadata['loaded_from_disk'] = bool(use_disk_cache)
    metadata['reused_from_cached_topk'] = None if not use_disk_cache else int((existing_metadata or {}).get('config', {}).get('topk', topk))
    artifact_paths = {
        'pickle_path': str(cache_path),
        'preview_path': str(preview_path),
        'metadata_path': str(metadata_path),
    }
    return retriever_cache, metadata, artifact_paths



def build_retriever_caches(dataset_name, train_df, dev_df, vocab_df, cfg):
    retriever_model = load_retriever_model(cfg['RETRIEVER_MODEL_NAME_OR_PATH'], device=cfg['DEVICE'])

    train_cache, train_cache_metadata, train_cache_paths = prepare_retriever_candidate_cache(
        dataset_name=dataset_name,
        split_name='train',
        entities_df=train_df,
        vocab_df=vocab_df,
        cfg=cfg,
        retriever_model=retriever_model,
        topk=cfg['TRAIN_CANDIDATE_POOL_SIZE'],
        mention_column='text',
    )
    dev_cache, dev_cache_metadata, dev_cache_paths = prepare_retriever_candidate_cache(
        dataset_name=dataset_name,
        split_name='dev',
        entities_df=dev_df,
        vocab_df=vocab_df,
        cfg=cfg,
        retriever_model=retriever_model,
        topk=cfg['DEV_CANDIDATE_POOL_SIZE'],
        mention_column='text',
    )

    del retriever_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return train_cache, dev_cache, {
        'train': train_cache_paths,
        'dev': dev_cache_paths,
    }, {
        'train': train_cache_metadata,
        'dev': dev_cache_metadata,
    }



def prepare_test_cache(dataset_name, test_df, vocab_df, cfg):
    retriever_model = load_retriever_model(cfg['RETRIEVER_MODEL_NAME_OR_PATH'], device=cfg['DEVICE'])
    test_cache, test_cache_metadata, test_cache_paths = prepare_retriever_candidate_cache(
        dataset_name=dataset_name,
        split_name='test',
        entities_df=test_df,
        vocab_df=vocab_df,
        cfg=cfg,
        retriever_model=retriever_model,
        topk=cfg['TEST_CANDIDATE_POOL_SIZE'],
        mention_column='text',
    )

    del retriever_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return test_cache, test_cache_metadata, test_cache_paths



def run_training(dataset_name, train_df, dev_df, vocab_df, cfg, candidate_text_map=None, candidate_artifact_paths=None, candidate_metadata=None):
    artifact_dir = get_artifact_dir(dataset_name)
    train_output_dir = artifact_dir / 'training'
    runtime_cfg = build_runtime_config(cfg)
    set_seed(runtime_cfg['SEED'])

    prepared_train_df, train_summary = prepare_mention_split(train_df, runtime_cfg, 'train')
    prepared_dev_df, dev_summary = prepare_mention_split(dev_df, runtime_cfg, 'dev')

    train_candidate_cache, dev_candidate_cache, retriever_cache_paths, retriever_cache_metadata = build_retriever_caches(
        dataset_name=dataset_name,
        train_df=train_df,
        dev_df=dev_df,
        vocab_df=vocab_df,
        cfg=runtime_cfg,
    )

    mention_column = (
        runtime_cfg['MENTION_CONTEXT'].get('CONTEXTUALIZED_MENTION_COLUMN', 'contextualized_mention_text')
        if runtime_cfg['MENTION_CONTEXT'].get('ENABLED', False)
        else 'text'
    )

    training_result = train_cross_encoder_reranker(
        train_df=prepared_train_df,
        dev_df=prepared_dev_df,
        vocab_df=vocab_df,
        cross_encoder_model_name=runtime_cfg['RERANKER_MODEL_NAME_OR_PATH'],
        output_dir=train_output_dir,
        cfg=runtime_cfg,
        train_candidate_cache=train_candidate_cache,
        dev_candidate_cache=dev_candidate_cache,
        mention_column=mention_column,
        candidate_text_map=candidate_text_map,
    )
    effective_cfg = build_effective_experiment_config(
        base_cfg=cfg,
        training_result=training_result,
        prepared_summaries={'train': train_summary, 'dev': dev_summary},
        candidate_artifact_paths=candidate_artifact_paths or {},
        candidate_metadata=candidate_metadata or {},
        retriever_cache_paths=retriever_cache_paths,
        retriever_cache_metadata=retriever_cache_metadata,
    )

    return training_result, effective_cfg, prepared_train_df, prepared_dev_df


## Prediction Utils


In [ ]:
def resolve_mention_column(cfg):
    runtime_cfg = build_runtime_config(cfg)
    return (
        runtime_cfg['MENTION_CONTEXT'].get('CONTEXTUALIZED_MENTION_COLUMN', 'contextualized_mention_text')
        if runtime_cfg['MENTION_CONTEXT'].get('ENABLED', False)
        else 'text'
    )



def load_best_model(effective_cfg):
    return load_cross_encoder_model_with_config(
        effective_cfg['TRAINING']['BEST_MODEL_DIR'],
        device=effective_cfg['DEVICE'],
        cfg=build_runtime_config(effective_cfg),
    )



def compute_dev_metrics_if_needed(dataset_name, data_df, vocab_df, cross_encoder_model, cfg, candidate_text_map=None):
    runtime_cfg = build_runtime_config(cfg)
    if runtime_cfg.get('EVAL_EVERY_EPOCH', True):
        return None, None

    dev_candidate_cache = build_retriever_candidate_cache(
        entities_df=data_df,
        vocab_df=vocab_df,
        retriever_model=load_retriever_model(runtime_cfg['RETRIEVER_MODEL_NAME_OR_PATH'], device=runtime_cfg['DEVICE']),
        mention_column='text',
        topk=runtime_cfg['DEV_CANDIDATE_POOL_SIZE'],
        query_batch_size=runtime_cfg['QUERY_BATCH_SIZE'],
        dense_vocab_batch_size=runtime_cfg['DENSE_VOCAB_BATCH_SIZE'],
        st_encode_batch_size=runtime_cfg['ST_ENCODE_BATCH_SIZE'],
        deduplicate_by_cui=runtime_cfg['DEDUPLICATE_BY_CUI'],
    )
    dev_predictions_df = rerank_from_candidate_cache(
        data_df=data_df,
        retriever_cache=dev_candidate_cache,
        cross_encoder_model=cross_encoder_model,
        mention_column=resolve_mention_column(cfg),
        candidate_text_map=candidate_text_map,
        batch_size=runtime_cfg['RERANK_BATCH_SIZE'],
        topk=runtime_cfg['DEV_RETURN_TOPK'],
    )
    dev_metrics = evaluate_dev_predictions(predictions_df=dev_predictions_df, data_df=data_df)
    return dev_predictions_df, dev_metrics



def predict_on_test(data_df, candidate_cache, cross_encoder_model, cfg, output_path, candidate_text_map=None):
    predictions_df = rerank_from_candidate_cache(
        data_df=data_df,
        retriever_cache=candidate_cache,
        cross_encoder_model=cross_encoder_model,
        mention_column=resolve_mention_column(cfg),
        candidate_text_map=candidate_text_map,
        batch_size=build_runtime_config(cfg)['RERANK_BATCH_SIZE'],
        topk=build_runtime_config(cfg)['TEST_RETURN_TOPK'],
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    predictions_df.to_csv(output_path, sep='	', index=False)
    return predictions_df, output_path


## MLflow Utils


In [ ]:
def flatten_config_for_mlflow(prefix, value):
    if isinstance(value, dict):
        flat = {}
        for key, nested_value in value.items():
            nested_prefix = f'{prefix}.{key}' if prefix else str(key)
            flat.update(flatten_config_for_mlflow(nested_prefix, nested_value))
        return flat
    if isinstance(value, (list, tuple)):
        return {prefix: str(list(value))}
    return {prefix: value}



def build_mlflow_params(dataset_name, base_cfg, effective_cfg):
    training_cfg = effective_cfg['TRAINING']
    model_cfg = effective_cfg['MODEL']
    retrieval_cfg = effective_cfg['RETRIEVAL']
    mention_context_cfg = effective_cfg['MENTION_CONTEXT']
    candidate_context_cfg = effective_cfg['CANDIDATE_CONTEXT']
    return {
        'dataset_name': dataset_name,
        'reranker_model_name_or_path': effective_cfg['RERANKER_MODEL_NAME_OR_PATH'],
        'retriever_model_name_or_path': effective_cfg['RETRIEVER_MODEL_NAME_OR_PATH'],
        'device': effective_cfg['DEVICE'],
        'trust_remote_code': bool(model_cfg.get('TRUST_REMOTE_CODE', False)),
        'local_files_only': bool(model_cfg.get('LOCAL_FILES_ONLY', False)),
        'model_torch_dtype': None if model_cfg.get('TORCH_DTYPE') is None else str(model_cfg.get('TORCH_DTYPE')),
        'seed': int(training_cfg['SEED']),
        'epochs': int(training_cfg['EPOCHS']),
        'train_batch_size': int(training_cfg['TRAIN_BATCH_SIZE']),
        'eval_batch_size': int(training_cfg['EVAL_BATCH_SIZE']),
        'learning_rate': float(training_cfg['LEARNING_RATE']),
        'weight_decay': float(training_cfg['WEIGHT_DECAY']),
        'warmup_ratio': float(training_cfg['WARMUP_RATIO']),
        'max_seq_length': int(training_cfg['MAX_SEQ_LENGTH']),
        'grad_accumulation_steps': int(training_cfg['GRAD_ACCUMULATION_STEPS']),
        'use_fp16': bool(training_cfg.get('USE_FP16', False)),
        'use_bf16': bool(training_cfg.get('USE_BF16', False)),
        'allow_tf32': bool(training_cfg.get('ALLOW_TF32', False)),
        'early_stopping_patience': int(training_cfg.get('EARLY_STOPPING_PATIENCE', 3)),
        'loss_name': str(training_cfg.get('LOSS_NAME', 'bce')),
        'lambdaloss_k': training_cfg.get('LAMBDALOSS_K'),
        'lambdaloss_weighting_scheme': training_cfg.get('LAMBDALOSS_WEIGHTING_SCHEME'),
        'lambdaloss_sigma': float(training_cfg.get('LAMBDALOSS_SIGMA', 1.0)),
        'lambdaloss_reduction_log': training_cfg.get('LAMBDALOSS_REDUCTION_LOG'),
        'lambdaloss_mini_batch_size': training_cfg.get('LAMBDALOSS_MINI_BATCH_SIZE'),
        'selection_metric': training_cfg['SELECTION_METRIC'],
        'best_epoch': int(training_cfg['BEST_EPOCH']),
        'num_train_lists': int(training_cfg['NUM_TRAIN_LISTS']),
        'num_train_pairs': int(training_cfg.get('NUM_TRAIN_PAIRS', 0)),
        'rerank_batch_size': int(training_cfg['RERANK_BATCH_SIZE']),
        'train_candidate_pool_size': int(retrieval_cfg['TRAIN_CANDIDATE_POOL_SIZE']),
        'dev_candidate_pool_size': int(retrieval_cfg['DEV_CANDIDATE_POOL_SIZE']),
        'dev_return_topk': int(retrieval_cfg['DEV_RETURN_TOPK']),
        'test_candidate_pool_size': int(retrieval_cfg['TEST_CANDIDATE_POOL_SIZE']),
        'test_return_topk': int(retrieval_cfg['TEST_RETURN_TOPK']),
        'query_batch_size': int(retrieval_cfg['QUERY_BATCH_SIZE']),
        'dense_vocab_batch_size': int(retrieval_cfg['DENSE_VOCAB_BATCH_SIZE']),
        'st_encode_batch_size': int(retrieval_cfg['ST_ENCODE_BATCH_SIZE']),
        'deduplicate_by_cui': bool(retrieval_cfg['DEDUPLICATE_BY_CUI']),
        'test_enrich_vocabulary': bool(retrieval_cfg.get('TEST_ENRICH_VOCABULARY', retrieval_cfg.get('ENRICH_VOCABULARY', False))),
        'mention_context_enabled': bool(mention_context_cfg['ENABLED']),
        'mention_context_mode': mention_context_cfg['MODE'],
        'mention_context_format': mention_context_cfg['FORMAT'],
        'candidate_context_enabled': bool(candidate_context_cfg['ENABLED']),
        'candidate_context_max_aliases': int(candidate_context_cfg['MAX_ALIASES']),
        'candidate_context_num_workers': int(candidate_context_cfg.get('NUM_WORKERS', 1)),
        'candidate_context_alias_length_threshold': candidate_context_cfg['METADATA'].get('threshold') if candidate_context_cfg.get('METADATA') else None,
    }



def log_run_to_mlflow(dataset_name, base_cfg, effective_cfg, dev_metrics, test_metrics, artifact_paths, training_result):
    experiment_name = f'{MLFLOW_EXPERIMENT_PREFIX}-{dataset_name.lower()}'
    mlflow.set_experiment(experiment_name)
    run_name = MLFLOW_RUN_NAME_TEMPLATE.format(dataset_name=dataset_name.lower())

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(build_mlflow_params(dataset_name, base_cfg, effective_cfg))
        mlflow_metrics = {}
        for metric_name, metric_value in dev_metrics.items():
            normalized_name = metric_name.lower().replace('@', '_at_')
            mlflow_metrics[f'dev_{normalized_name}'] = float(metric_value)
        for metric_name, metric_value in test_metrics.items():
            normalized_name = metric_name.lower().replace('@', '_at_')
            mlflow_metrics[f'test_{normalized_name}'] = float(metric_value)
        mlflow.log_metrics(mlflow_metrics)

        history_df = training_result['history_df'].copy()
        if not history_df.empty:
            if 'step' not in history_df.columns:
                history_df['step'] = range(1, len(history_df) + 1)
            dev_acc_history_columns = [
                column_name
                for column_name in history_df.columns
                if isinstance(column_name, str) and column_name.startswith('dev_Acc@')
            ]
            for _, row in history_df.iterrows():
                step = int(row['step']) if 'step' in row and row['step'] == row['step'] else None
                if 'loss' in row and row['loss'] == row['loss']:
                    mlflow.log_metric('train_loss', float(row['loss']), step=step)
                if 'eval_loss' in row and row['eval_loss'] == row['eval_loss']:
                    mlflow.log_metric('dev_loss', float(row['eval_loss']), step=step)
            if dev_acc_history_columns and 'epoch_int' in history_df.columns:
                dev_acc_history_df = history_df.dropna(subset=['epoch_int']).copy()
                dev_acc_history_df = dev_acc_history_df.dropna(how='all', subset=dev_acc_history_columns)
                if not dev_acc_history_df.empty:
                    dev_acc_history_df['epoch_int'] = dev_acc_history_df['epoch_int'].astype(int)
                    dev_acc_history_df = dev_acc_history_df.sort_values(['epoch_int', 'step'], kind='stable')
                    dev_acc_history_df = dev_acc_history_df.groupby('epoch_int', as_index=False)[dev_acc_history_columns].last()
                    for _, row in dev_acc_history_df.iterrows():
                        epoch_step = int(row['epoch_int'])
                        for column_name in dev_acc_history_columns:
                            metric_value = row[column_name]
                            if metric_value == metric_value:
                                normalized_name = column_name.removeprefix('dev_').lower().replace('@', '_at_')
                                mlflow.log_metric(f'dev_{normalized_name}_history', float(metric_value), step=epoch_step)

        artifact_dir_map = {
            'best_model': 'model',
            'prepared_train_mentions': 'prepared_data',
            'prepared_dev_mentions': 'prepared_data',
            'prepared_test_mentions': 'prepared_data',
            'candidate_context_preview': 'prepared_data',
            'candidate_context_metadata': 'prepared_data',
            'train_retriever_cache_preview': 'prepared_data',
            'train_retriever_cache_metadata': 'prepared_data',
            'dev_retriever_cache_preview': 'prepared_data',
            'dev_retriever_cache_metadata': 'prepared_data',
            'test_retriever_cache_preview': 'prepared_data',
            'test_retriever_cache_metadata': 'prepared_data',
            'test_predictions': 'predictions',
            'test_metrics': 'metrics',
            'metrics_summary': 'metrics',
            'base_config': 'configs',
            'effective_config': 'configs',
            'training_history': 'training',
            'train_examples': 'training',
        }
        for artifact_name, artifact_path in artifact_paths.items():
            target_dir = artifact_dir_map.get(artifact_name)
            if target_dir is None:
                continue
            if Path(artifact_path).is_dir():
                mlflow.log_artifacts(artifact_path, artifact_path=target_dir)
            else:
                mlflow.log_artifact(artifact_path, artifact_path=target_dir)

        return mlflow.active_run().info.run_id


## Artifacts Utils


In [ ]:
def save_local_artifacts(
    dataset_name,
    base_cfg,
    effective_cfg,
    prepared_train_df,
    prepared_dev_df,
    prepared_test_df,
    candidate_context_df,
    candidate_context_metadata,
    dev_metrics,
    test_predictions_df,
    test_metrics,
    training_result,
    test_predictions_path=None,
):
    artifact_dir = get_artifact_dir(dataset_name)

    if test_predictions_path is None:
        test_predictions_path = artifact_dir / 'test_predictions.tsv'
    else:
        test_predictions_path = Path(test_predictions_path)

    test_metrics_path = artifact_dir / 'test_metrics.json'
    metrics_table_path = artifact_dir / 'metrics_summary.tsv'
    training_history_path = artifact_dir / 'training_history.tsv'
    train_examples_path = artifact_dir / 'train_examples.tsv'
    base_config_path = artifact_dir / 'base_config.json'
    effective_config_path = artifact_dir / 'effective_config.json'
    prepared_train_path = artifact_dir / 'prepared_train_mentions.tsv'
    prepared_dev_path = artifact_dir / 'prepared_dev_mentions.tsv'
    prepared_test_path = artifact_dir / 'prepared_test_mentions.tsv'
    best_model_path = artifact_dir / 'best_model'
    candidate_context_preview_path = artifact_dir / 'candidate_context_preview.tsv'
    candidate_context_metadata_path = artifact_dir / 'candidate_context_runtime_metadata.json'

    prepared_train_df.to_csv(prepared_train_path, sep='	', index=False)
    prepared_dev_df.to_csv(prepared_dev_path, sep='	', index=False)
    prepared_test_df.to_csv(prepared_test_path, sep='	', index=False)
    if not test_predictions_path.exists():
        test_predictions_df.to_csv(test_predictions_path, sep='	', index=False)

    if candidate_context_df is not None:
        serializable_candidate_df = candidate_context_df.copy()
        if 'selected_aliases' in serializable_candidate_df.columns:
            serializable_candidate_df['selected_aliases'] = serializable_candidate_df['selected_aliases'].map(
                lambda values: ' | '.join(values) if isinstance(values, list) else str(values)
            )
        if 'selected_aliases_normalized' in serializable_candidate_df.columns:
            serializable_candidate_df['selected_aliases_normalized'] = serializable_candidate_df['selected_aliases_normalized'].map(
                lambda values: ' | '.join(values) if isinstance(values, list) else str(values)
            )
        if 'languages' in serializable_candidate_df.columns:
            serializable_candidate_df['languages'] = serializable_candidate_df['languages'].map(
                lambda values: ','.join(values) if isinstance(values, list) else str(values)
            )
        serializable_candidate_df.head(200).to_csv(candidate_context_preview_path, sep='	', index=False)
    if best_model_path.exists():
        shutil.rmtree(best_model_path)
    shutil.copytree(Path(effective_cfg['TRAINING']['BEST_MODEL_DIR']), best_model_path)

    save_json(candidate_context_metadata, candidate_context_metadata_path)
    save_json(test_metrics, test_metrics_path)

    pd.DataFrame([
        {'split': 'dev', **dev_metrics},
        {'split': 'test', **test_metrics},
    ]).to_csv(metrics_table_path, sep='	', index=False)

    training_result['history_df'].to_csv(training_history_path, sep='	', index=False)
    training_result['train_examples_df'].to_csv(train_examples_path, sep='	', index=False)

    save_json(base_cfg, base_config_path)
    save_json(effective_cfg, effective_config_path)

    artifact_paths = {
        'artifact_dir': str(artifact_dir),
        'best_model': str(best_model_path),
        'prepared_train_mentions': str(prepared_train_path),
        'prepared_dev_mentions': str(prepared_dev_path),
        'prepared_test_mentions': str(prepared_test_path),
        'candidate_context_preview': str(candidate_context_preview_path),
        'candidate_context_metadata': str(candidate_context_metadata_path),
        'test_predictions': str(test_predictions_path),
        'test_metrics': str(test_metrics_path),
        'metrics_summary': str(metrics_table_path),
        'training_history': str(training_history_path),
        'train_examples': str(train_examples_path),
        'base_config': str(base_config_path),
        'effective_config': str(effective_config_path),
    }

    retriever_cache_artifacts = effective_cfg.get('RETRIEVAL', {}).get('CACHE_ARTIFACTS', {})
    for split_name, split_paths in retriever_cache_artifacts.items():
        if not isinstance(split_paths, dict):
            continue
        preview_path = split_paths.get('preview_path')
        metadata_path = split_paths.get('metadata_path')
        if preview_path:
            artifact_paths[f'{split_name}_retriever_cache_preview'] = str(preview_path)
        if metadata_path:
            artifact_paths[f'{split_name}_retriever_cache_metadata'] = str(metadata_path)

    return artifact_paths


## RU Experiment


In [ ]:
RU_CONFIG = build_base_experiment_config(
    reranker_model_name_or_path='models/ru-bce-bionnel-cross-encoder-dict-pretrain',
    dataset_name='ru',
)

RU_CONFIG['RETRIEVER_MODEL_NAME_OR_PATH'] = 'models/ru-dense-finetuning-infonce-no-hard-negatives'
RU_CONFIG


In [ ]:
ru_vocab = enrich_vocab_with_oov_train_dev_terms(vocab.copy(), RU_ENTITIES_FOR_VOCAB_ENRICHMENT)
ru_vocab = prepare_experiment_vocab(ru_vocab, RU_ENTITIES_FOR_VOCAB_ENRICHMENT, RU_CONFIG['RETRIEVAL'])
ru_candidate_context_vocab = prepare_candidate_context_vocab(
    enrich_vocab_with_oov_train_dev_terms(
        raw_vocab.copy(),
        RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
        text_column='raw_text',
    ),
    RU_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
    RU_CONFIG,
)


In [ ]:
(
    ru_candidate_context_df,
    ru_candidate_context_metadata,
    ru_candidate_context_artifact_paths,
    ru_candidate_text_map,
) = prepare_candidate_context_cache_for_experiment(
    vocab_df=ru_candidate_context_vocab,
    dataset_name='ru',
    cfg=build_runtime_config(RU_CONFIG),
)
ru_candidate_context_artifact_paths, ru_candidate_context_metadata, ru_candidate_context_df.head(3) if ru_candidate_context_df is not None else 'Candidate context disabled' 


In [ ]:
(
    ru_training_result,
    RU_EFFECTIVE_CONFIG,
    ru_prepared_train_df,
    ru_prepared_dev_df,
) = run_training(
    dataset_name='ru',
    train_df=ru_data_train,
    dev_df=ru_data_dev,
    vocab_df=ru_vocab,
    cfg=RU_CONFIG,
    candidate_text_map=ru_candidate_text_map,
    candidate_artifact_paths=ru_candidate_context_artifact_paths,
    candidate_metadata=ru_candidate_context_metadata,
)


In [ ]:
ru_best_cross_encoder = load_best_model(RU_EFFECTIVE_CONFIG)
ru_dev_metrics = dict(ru_training_result['best_metrics'])
if not build_runtime_config(RU_EFFECTIVE_CONFIG).get('EVAL_EVERY_EPOCH', True) and not ru_dev_metrics:
    _, ru_dev_metrics = compute_dev_metrics_if_needed(
        dataset_name='ru',
        data_df=ru_prepared_dev_df,
        vocab_df=ru_vocab,
        cross_encoder_model=ru_best_cross_encoder,
        cfg=RU_EFFECTIVE_CONFIG,
        candidate_text_map=ru_candidate_text_map,
    )
RU_EFFECTIVE_CONFIG, ru_dev_metrics


In [ ]:
ru_test_cfg = build_test_retrieval_config(RU_EFFECTIVE_CONFIG)
ru_test_vocab = build_test_retrieval_vocab(vocab, RU_ENTITIES_FOR_VOCAB_ENRICHMENT, RU_EFFECTIVE_CONFIG, dataset_name='ru')
ru_prepared_test_df, ru_test_summary = prepare_mention_split(ru_data_test, build_runtime_config(ru_test_cfg), 'test')
ru_test_candidate_cache, ru_test_cache_metadata, ru_test_cache_paths = prepare_test_cache('ru', ru_data_test, ru_test_vocab, build_runtime_config(ru_test_cfg))
RU_EFFECTIVE_CONFIG['MENTION_CONTEXT']['PREPARED_SPLIT_SUMMARIES']['test'] = ru_test_summary
RU_EFFECTIVE_CONFIG['RETRIEVAL']['CACHE_ARTIFACTS']['test'] = ru_test_cache_paths
RU_EFFECTIVE_CONFIG['RETRIEVAL']['CACHE_METADATA']['test'] = ru_test_cache_metadata
ru_test_predictions_df, ru_test_predictions_path = predict_on_test(
    data_df=ru_prepared_test_df,
    candidate_cache=ru_test_candidate_cache,
    cross_encoder_model=ru_best_cross_encoder,
    cfg=RU_EFFECTIVE_CONFIG,
    output_path=get_artifact_dir('ru') / 'test_predictions.tsv',
    candidate_text_map=ru_candidate_text_map,
)
ru_test_metrics = evaluate_dev_predictions(predictions_df=ru_test_predictions_df, data_df=ru_data_test)
ru_test_metrics


In [ ]:
ru_artifact_paths = save_local_artifacts(
    dataset_name='ru',
    base_cfg=RU_CONFIG,
    effective_cfg=RU_EFFECTIVE_CONFIG,
    prepared_train_df=ru_prepared_train_df,
    prepared_dev_df=ru_prepared_dev_df,
    prepared_test_df=ru_prepared_test_df,
    candidate_context_df=ru_candidate_context_df,
    candidate_context_metadata=ru_candidate_context_metadata,
    dev_metrics=ru_dev_metrics,
    test_predictions_df=ru_test_predictions_df,
    test_metrics=ru_test_metrics,
    training_result=ru_training_result,
    test_predictions_path=ru_test_predictions_path,
)
ru_artifact_paths


In [ ]:
ru_mlflow_run_id = log_run_to_mlflow(
    dataset_name='ru',
    base_cfg=RU_CONFIG,
    effective_cfg=RU_EFFECTIVE_CONFIG,
    dev_metrics=ru_dev_metrics,
    test_metrics=ru_test_metrics,
    training_result=ru_training_result,
    artifact_paths=ru_artifact_paths,
)
ru_artifact_paths, ru_mlflow_run_id


In [ ]:
del ru_best_cross_encoder
if 'ru_dev_candidate_cache' in globals():
    del ru_dev_candidate_cache
if 'ru_test_candidate_cache' in globals():
    del ru_test_candidate_cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## EN Experiment


In [ ]:
EN_CONFIG = build_base_experiment_config(
    reranker_model_name_or_path='andorei/BERGAMOT-multilingual-GAT',
    dataset_name='en',
)
EN_CONFIG['RETRIEVER_MODEL_NAME_OR_PATH'] = 'andorei/BERGAMOT-multilingual-GAT'
EN_CONFIG


In [ ]:
en_vocab = enrich_vocab_with_oov_train_dev_terms(
    vocab.copy(),
    EN_ENTITIES_FOR_VOCAB_ENRICHMENT,
    lang_value="EN",
)
en_vocab = prepare_experiment_vocab(
    en_vocab,
    EN_ENTITIES_FOR_VOCAB_ENRICHMENT,
    EN_CONFIG['RETRIEVAL'],
    lang_value="EN",
)
en_vocab = filter_vocab_for_dataset_language(en_vocab, "en")
en_candidate_context_vocab = filter_vocab_for_dataset_language(raw_vocab.copy(), "en")
en_candidate_context_vocab = enrich_vocab_with_oov_train_dev_terms(
    en_candidate_context_vocab,
    EN_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
    text_column='raw_text',
    lang_value="EN",
)
en_candidate_context_vocab = prepare_candidate_context_vocab(
    en_candidate_context_vocab,
    EN_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
    EN_CONFIG,
    lang_value="EN",
)


In [ ]:
(
    en_candidate_context_df,
    en_candidate_context_metadata,
    en_candidate_context_artifact_paths,
    en_candidate_text_map,
) = prepare_candidate_context_cache_for_experiment(
    vocab_df=en_candidate_context_vocab,
    dataset_name='en',
    cfg=build_runtime_config(EN_CONFIG),
)
en_candidate_context_artifact_paths, en_candidate_context_metadata, en_candidate_context_df.head(3) if en_candidate_context_df is not None else 'Candidate context disabled' 


In [ ]:
(
    en_training_result,
    EN_EFFECTIVE_CONFIG,
    en_prepared_train_df,
    en_prepared_dev_df,
) = run_training(
    dataset_name='en',
    train_df=en_data_train,
    dev_df=en_data_dev,
    vocab_df=en_vocab,
    cfg=EN_CONFIG,
    candidate_text_map=en_candidate_text_map,
    candidate_artifact_paths=en_candidate_context_artifact_paths,
    candidate_metadata=en_candidate_context_metadata,
)


In [ ]:
en_best_cross_encoder = load_best_model(EN_EFFECTIVE_CONFIG)
en_dev_metrics = dict(en_training_result['best_metrics'])
if not build_runtime_config(EN_EFFECTIVE_CONFIG).get('EVAL_EVERY_EPOCH', True) and not en_dev_metrics:
    _, en_dev_metrics = compute_dev_metrics_if_needed(
        dataset_name='en',
        data_df=en_prepared_dev_df,
        vocab_df=en_vocab,
        cross_encoder_model=en_best_cross_encoder,
        cfg=EN_EFFECTIVE_CONFIG,
        candidate_text_map=en_candidate_text_map,
    )
EN_EFFECTIVE_CONFIG, en_dev_metrics


In [ ]:
en_prepared_test_df, en_test_summary = prepare_mention_split(en_data_test, build_runtime_config(EN_EFFECTIVE_CONFIG), 'test')
en_test_candidate_cache, en_test_cache_metadata, en_test_cache_paths = prepare_test_cache('en', en_data_test, en_vocab, build_runtime_config(EN_EFFECTIVE_CONFIG))
EN_EFFECTIVE_CONFIG['MENTION_CONTEXT']['PREPARED_SPLIT_SUMMARIES']['test'] = en_test_summary
EN_EFFECTIVE_CONFIG['RETRIEVAL']['CACHE_ARTIFACTS']['test'] = en_test_cache_paths
EN_EFFECTIVE_CONFIG['RETRIEVAL']['CACHE_METADATA']['test'] = en_test_cache_metadata
en_test_predictions_df, en_test_predictions_path = predict_on_test(
    data_df=en_prepared_test_df,
    candidate_cache=en_test_candidate_cache,
    cross_encoder_model=en_best_cross_encoder,
    cfg=EN_EFFECTIVE_CONFIG,
    output_path=get_artifact_dir('en') / 'test_predictions.tsv',
    candidate_text_map=en_candidate_text_map,
)
en_test_metrics = evaluate_dev_predictions(predictions_df=en_test_predictions_df, data_df=en_data_test)
en_test_metrics


In [ ]:
en_artifact_paths = save_local_artifacts(
    dataset_name='en',
    base_cfg=EN_CONFIG,
    effective_cfg=EN_EFFECTIVE_CONFIG,
    prepared_train_df=en_prepared_train_df,
    prepared_dev_df=en_prepared_dev_df,
    prepared_test_df=en_prepared_test_df,
    candidate_context_df=en_candidate_context_df,
    candidate_context_metadata=en_candidate_context_metadata,
    dev_metrics=en_dev_metrics,
    test_predictions_df=en_test_predictions_df,
    test_metrics=en_test_metrics,
    training_result=en_training_result,
    test_predictions_path=en_test_predictions_path,
)
en_artifact_paths


In [ ]:
en_mlflow_run_id = log_run_to_mlflow(
    dataset_name='en',
    base_cfg=EN_CONFIG,
    effective_cfg=EN_EFFECTIVE_CONFIG,
    dev_metrics=en_dev_metrics,
    test_metrics=en_test_metrics,
    training_result=en_training_result,
    artifact_paths=en_artifact_paths,
)
en_artifact_paths, en_mlflow_run_id


In [ ]:
del en_best_cross_encoder
if 'en_dev_candidate_cache' in globals():
    del en_dev_candidate_cache
if 'en_test_candidate_cache' in globals():
    del en_test_candidate_cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Bilingual Experiment


In [ ]:
BILINGUAL_CONFIG = build_base_experiment_config(
    reranker_model_name_or_path='andorei/BERGAMOT-multilingual-GAT',
    dataset_name='bilingual',
)
BILINGUAL_CONFIG['RETRIEVER_MODEL_NAME_OR_PATH'] = 'andorei/BERGAMOT-multilingual-GAT'
BILINGUAL_CONFIG


In [ ]:
bilingual_vocab = enrich_vocab_with_oov_train_dev_terms(vocab.copy(), BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT)
bilingual_vocab = prepare_experiment_vocab(
    bilingual_vocab,
    BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT,
    BILINGUAL_CONFIG['RETRIEVAL'],
)
bilingual_candidate_context_vocab = prepare_candidate_context_vocab(
    enrich_vocab_with_oov_train_dev_terms(
        raw_vocab.copy(),
        BILINGUAL_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
        text_column='raw_text',
    ),
    BILINGUAL_RAW_ENTITIES_FOR_CANDIDATE_CONTEXT_ENRICHMENT,
    BILINGUAL_CONFIG,
)


In [ ]:
(
    bilingual_candidate_context_df,
    bilingual_candidate_context_metadata,
    bilingual_candidate_context_artifact_paths,
    bilingual_candidate_text_map,
) = prepare_candidate_context_cache_for_experiment(
    vocab_df=bilingual_candidate_context_vocab,
    dataset_name='bilingual',
    cfg=build_runtime_config(BILINGUAL_CONFIG),
)
bilingual_candidate_context_artifact_paths, bilingual_candidate_context_metadata, bilingual_candidate_context_df.head(3) if bilingual_candidate_context_df is not None else 'Candidate context disabled' 


In [ ]:
(
    bilingual_training_result,
    BILINGUAL_EFFECTIVE_CONFIG,
    bilingual_prepared_train_df,
    bilingual_prepared_dev_df,
) = run_training(
    dataset_name='bilingual',
    train_df=bilingual_data_train,
    dev_df=bilingual_data_dev,
    vocab_df=bilingual_vocab,
    cfg=BILINGUAL_CONFIG,
    candidate_text_map=bilingual_candidate_text_map,
    candidate_artifact_paths=bilingual_candidate_context_artifact_paths,
    candidate_metadata=bilingual_candidate_context_metadata,
)


In [ ]:
bilingual_best_cross_encoder = load_best_model(BILINGUAL_EFFECTIVE_CONFIG)
bilingual_dev_metrics = dict(bilingual_training_result['best_metrics'])
if not build_runtime_config(BILINGUAL_EFFECTIVE_CONFIG).get('EVAL_EVERY_EPOCH', True) and not bilingual_dev_metrics:
    _, bilingual_dev_metrics = compute_dev_metrics_if_needed(
        dataset_name='bilingual',
        data_df=bilingual_prepared_dev_df,
        vocab_df=bilingual_vocab,
        cross_encoder_model=bilingual_best_cross_encoder,
        cfg=BILINGUAL_EFFECTIVE_CONFIG,
        candidate_text_map=bilingual_candidate_text_map,
    )
BILINGUAL_EFFECTIVE_CONFIG, bilingual_dev_metrics


In [ ]:
bilingual_prepared_test_df, bilingual_test_summary = prepare_mention_split(bilingual_data_test, build_runtime_config(BILINGUAL_EFFECTIVE_CONFIG), 'test')
bilingual_test_candidate_cache, bilingual_test_cache_metadata, bilingual_test_cache_paths = prepare_test_cache('bilingual', bilingual_data_test, bilingual_vocab, build_runtime_config(BILINGUAL_EFFECTIVE_CONFIG))
BILINGUAL_EFFECTIVE_CONFIG['MENTION_CONTEXT']['PREPARED_SPLIT_SUMMARIES']['test'] = bilingual_test_summary
BILINGUAL_EFFECTIVE_CONFIG['RETRIEVAL']['CACHE_ARTIFACTS']['test'] = bilingual_test_cache_paths
BILINGUAL_EFFECTIVE_CONFIG['RETRIEVAL']['CACHE_METADATA']['test'] = bilingual_test_cache_metadata
bilingual_test_predictions_df, bilingual_test_predictions_path = predict_on_test(
    data_df=bilingual_prepared_test_df,
    candidate_cache=bilingual_test_candidate_cache,
    cross_encoder_model=bilingual_best_cross_encoder,
    cfg=BILINGUAL_EFFECTIVE_CONFIG,
    output_path=get_artifact_dir('bilingual') / 'test_predictions.tsv',
    candidate_text_map=bilingual_candidate_text_map,
)
bilingual_test_metrics = evaluate_dev_predictions(predictions_df=bilingual_test_predictions_df, data_df=bilingual_data_test)
bilingual_test_metrics


In [ ]:
bilingual_artifact_paths = save_local_artifacts(
    dataset_name='bilingual',
    base_cfg=BILINGUAL_CONFIG,
    effective_cfg=BILINGUAL_EFFECTIVE_CONFIG,
    prepared_train_df=bilingual_prepared_train_df,
    prepared_dev_df=bilingual_prepared_dev_df,
    prepared_test_df=bilingual_prepared_test_df,
    candidate_context_df=bilingual_candidate_context_df,
    candidate_context_metadata=bilingual_candidate_context_metadata,
    dev_metrics=bilingual_dev_metrics,
    test_predictions_df=bilingual_test_predictions_df,
    test_metrics=bilingual_test_metrics,
    training_result=bilingual_training_result,
    test_predictions_path=bilingual_test_predictions_path,
)
bilingual_artifact_paths


In [ ]:
bilingual_mlflow_run_id = log_run_to_mlflow(
    dataset_name='bilingual',
    base_cfg=BILINGUAL_CONFIG,
    effective_cfg=BILINGUAL_EFFECTIVE_CONFIG,
    dev_metrics=bilingual_dev_metrics,
    test_metrics=bilingual_test_metrics,
    training_result=bilingual_training_result,
    artifact_paths=bilingual_artifact_paths,
)
bilingual_artifact_paths, bilingual_mlflow_run_id


In [ ]:
del bilingual_best_cross_encoder
if 'bilingual_dev_candidate_cache' in globals():
    del bilingual_dev_candidate_cache
if 'bilingual_test_candidate_cache' in globals():
    del bilingual_test_candidate_cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
